# Alternative Data

## Download the Data

- **Purpose:** Download and normalize the observed AAPL news used by the alternative-data workflow.
- **Settings:** `symbol=AAPL`; `period=2025`; `timezone=UTC`; source URLs use the AAPL-only Benzinga News and Analyst Ratings policy.
- **Data:** The input is observed AAPL news, and the output is the normalized Parquet cache.
- **Decision:** Reuse the cache only when it matches the requested symbol and period, and persist only the filtered project schema.

In [1]:
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.preprocessing.alternative_data import (
    filter_aapl_news_and_analyst_ratings,
    save_alpaca_news,
)

PROJECT_ROOT = Path.cwd().resolve().parents[1]
news_path = PROJECT_ROOT / "data/research_data/alternative/data/aapl_2025-01-01_2025-12-31.parquet"

if not news_path.is_file():
    save_alpaca_news(
        symbols=["AAPL"],
        start=datetime(2025, 1, 1, tzinfo=timezone.utc),
        end=datetime(2025, 12, 31, 23, 59, 59, 999999, tzinfo=timezone.utc),
        output_path=news_path,
    )

alternative_data = pd.read_parquet(news_path)
alternative_data["created_at"] = pd.to_datetime(alternative_data["created_at"], utc=True)
alternative_data["updated_at"] = pd.to_datetime(alternative_data["updated_at"], utc=True)

alternative_data = filter_aapl_news_and_analyst_ratings(alternative_data)
alternative_data.to_parquet(news_path, index=False)
news_path

PosixPath('/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/research_data/alternative/data/aapl_2025-01-01_2025-12-31.parquet')

## Take a Quick Look at the Data Structure

- **Purpose:** Inspect example rows, dtypes, source coverage, identifier ranges, and the filtered sample shape.
- **Settings:** No analytical parameters; read-only inspection.
- **Data:** Inspect the cached normalized AAPL news artifact.
- **Decision:** Leave the cached schema and retained rows unchanged.

In [2]:
alternative_data.head()

,id,headline,source,url,summary,created_at,updated_at,symbols,author,content
0,42767369,'CPCS Secures Agreement With Apple For Enhance...,benzinga,https://www.benzinga.com/news/25/01/42767369/c...,,2025-01-02 15:32:15+00:00,2025-01-02 15:32:15+00:00,AAPL,Benzinga Newsdesk,
1,42773823,Some Investors Are Selling Apple Stock Thursda...,benzinga,https://www.benzinga.com/news/global/25/01/427...,Apple Inc (NASDAQ:AAPL) shares are trading low...,2025-01-02 19:25:09+00:00,2025-01-02 19:25:10+00:00,AAPL,Adam Eckert,<p><strong>Apple Inc</strong> (NASDAQ:<a class...
2,42780075,Apple Settles $95 Million Lawsuit Over Siri Pr...,benzinga,https://www.benzinga.com/news/global/25/01/427...,Apple Inc. has agreed to pay $95 million to se...,2025-01-03 03:59:46+00:00,2025-01-03 03:59:47+00:00,AAPL,Kaustubh Bagalkote,<p><strong>Apple Inc. </strong>(NASDAQ:<a clas...
3,42784735,"B of A Securities Maintains Buy on Apple, Main...",benzinga,https://www.benzinga.com/news/25/01/42784735/b...,,2025-01-03 13:57:22+00:00,2025-01-03 13:57:22+00:00,AAPL,Benzinga Newsdesk,
4,42788545,"Bernstein Maintains Outperform on Apple, Raise...",benzinga,https://www.benzinga.com/news/25/01/42788545/b...,,2025-01-03 15:30:38+00:00,2025-01-03 15:30:39+00:00,AAPL,Benzinga Newsdesk,


In [3]:
alternative_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype              
---  ------      --------------  -----              
 0   id          395 non-null    int64              
 1   headline    395 non-null    str                
 2   source      395 non-null    str                
 3   url         395 non-null    str                
 4   summary     395 non-null    str                
 5   created_at  395 non-null    datetime64[us, UTC]
 6   updated_at  395 non-null    datetime64[us, UTC]
 7   symbols     395 non-null    str                
 8   author      395 non-null    str                
 9   content     395 non-null    str                
dtypes: datetime64[us, UTC](2), int64(1), str(7)
memory usage: 425.4 KB


In [4]:
alternative_data["source"].value_counts(dropna=False)

source
benzinga    395
Name: count, dtype: int64

In [5]:
alternative_data[["id"]].describe()

,id
count,3.950000e+02
mean,4.631025e+07
std,1.877257e+06
min,4.276737e+07
25%,4.486528e+07
50%,4.631357e+07
75%,4.777676e+07
max,4.965089e+07


In [6]:
alternative_data[["id"]].hist(bins=50, figsize=(6, 4))
plt.tight_layout()
plt.show()

/var/folders/1z/bcvql7210c77v6rjkswpzsyr0000gn/T/ipykernel_61139/3040956676.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
